# Quant Research Lab: 100 strategies on real data

This notebook runs the full lab from the `crunchhamburger/Backtest` repository on **real Yahoo Finance prices** and FRED T-bill rates.

**What it does:** downloads and validates data for ~35 tickers, runs the frozen 100-strategy registry across broad ETFs, sector ETFs, semiconductor ETFs and semiconductor stocks (10 bps per side, signal at the close, fill at the next open, T-bill cash), then walk-forward tests, multiple-testing statistics, the perfect-hindsight oracle and hypotheses H1-H10, and writes Excel/PDF reports.

Run cells in order (**Runtime -> Run all**). The full run takes roughly 10-20 minutes on a standard Colab runtime. If Yahoo rate-limits the download, wait and re-run the data cell: cached files are reused, and nothing is ever fabricated.

In [ ]:
BRANCH = "claude/dazzling-sagan-hc9tqf"
!git clone -q --branch $BRANCH https://github.com/crunchhamburger/Backtest.git lab 2>/dev/null || (cd lab && git pull -q)
%cd lab
!pip -q install -r requirements.txt

## 1. Automated tests (look-ahead, execution timing, costs, ledger, oracle, registry freeze)

In [ ]:
!python -m pytest -q tests -p no:cacheprovider

## 2. Download and validate data
Raw bars are cached under `data/cache/`. The validation table lists dividend/split checks, calendar gaps and any problems.

In [ ]:
import sys; sys.path.insert(0, ".")
import pandas as pd
from lab_config import load_config
from data_pipeline import load_market_data, data_inventory
cfg = load_config()
md = load_market_data(cfg)          # downloads once, then reads the cache
display(data_inventory(md))
display(md.validation[md.validation.severity != "info"])

## 3. Run every experiment
Use `--universes A_broad` or `--strategies F01_01,F01_02` for a quicker partial run; `--cost-bps 20` changes the base cost.

In [ ]:
!python run_experiments.py --offline

## 4. Results

In [ ]:
from IPython.display import Image, display
out = "outputs"
lb = pd.read_excel(f"{out}/Strategy_Leaderboard.xlsx", sheet_name="Hypotheses", skiprows=0)
display(lb[["id", "verdict", "evidence"]])
fam = pd.read_excel(f"{out}/Strategy_Leaderboard.xlsx", sheet_name="Family_Summary")
display(fam.sort_values("median_oos_sharpe_diff", ascending=False).head(20))
for f in ["family_comparison", "family_by_instrument_heatmap", "spy_original_six", "spy_sma1030_focus",
          "walk_forward_is_vs_oos", "oracle_SPY", "cost_sensitivity"]:
    display(Image(f"{out}/figures/{f}.png"))

## 5. Save outputs
Download the workbooks and PDFs, or copy them to Google Drive.

In [ ]:
!zip -qr lab_outputs.zip outputs data/cache
from google.colab import files
files.download("lab_outputs.zip")